# Enhanced IPL Match Chunking and RAG Ingestion

This notebook implements a detailed chunking strategy to parse raw ball-by-ball IPL match data from JSON files (`ipl-2008-2026/*.json`) into specialized cricket knowledge chunks. These chunks are designed for optimal retrieval accuracy under semantic search.

We generate seven specialized chunk types per match:
1. **Match Summary Chunk** (Match result, venue, player of the match, high-level summary)
2. **Team Innings Summary Chunk** (Runs, wickets, overs, top-scorers for each innings)
3. **Player Performance Chunk** (Individual batting and bowling statistics)
4. **Partnership Chunk** (Runs added, wicket fell, batsmen involved)
5. **Wicket Chunk** (Over, batsman, bowler, dismissal type, fielder)
6. **Milestone Chunk** (Centuries, half-centuries, 3/5-wicket hauls)
7. **Match Narrative Chunk** (Natural language paragraphs representing the overall match progression)

We then ingest all chunks into ChromaDB with a curated metadata dictionary to support precise filtering.

In [2]:
import os
import glob
import json
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion

# Load environment variables
load_dotenv('../.env')
if not os.getenv('OPENAI_API_KEY'):
    load_dotenv()

print("Environment setup complete. OpenAI API Key available:", os.getenv("OPENAI_API_KEY") is not None)

embedding_model_name = "text-embedding-3-large"

openai = OpenAI()

Environment setup complete. OpenAI API Key available: True


In [3]:
def get_season_year(season_str):
    season_str = str(season_str)
    mapping = {
        "2007/08": "2008",
        "2009/10": "2010",
        "2020/21": "2020"
    }
    return mapping.get(season_str, season_str)

def get_sort_key(item):
    filepath, data = item
    info = data.get('info', {})
    dates = info.get('dates', [])
    date_str = dates[0] if dates else '9999-12-31'
    
    event = info.get('event', {})
    match_num = event.get('match_number', 999)
    if isinstance(match_num, str):
        if 'final' in match_num.lower():
            match_num_val = 200
        elif 'semi' in match_num.lower():
            match_num_val = 190
        elif 'qualifier' in match_num.lower():
            match_num_val = 180
        elif 'eliminator' in match_num.lower():
            match_num_val = 175
        else:
            match_num_val = 150
    else:
        match_num_val = int(match_num) if match_num is not None else 999
        
    filename = filepath.split('/')[-1]
    return (date_str, match_num_val, filename)

In [4]:


# Determine dataset directory dynamically
if os.path.exists('ipl-2008-2026') and glob.glob('ipl-2008-2026/*.json'):
    dataset_dir = 'ipl-2008-2026'
elif os.path.exists('ipl-2008') and glob.glob('ipl-2008/*.json'):
    dataset_dir = 'ipl-2008'
else:
    dataset_dir = 'dataset'

print(f"Using dataset directory: {dataset_dir}")
json_files = glob.glob(f'{dataset_dir}/*.json')
all_matches = []

for f in json_files:
    with open(f, 'r') as file:
        try:
            data = json.load(file)
            all_matches.append((f, data))
        except json.JSONDecodeError:
            continue

# Sort matches chronologically
all_matches.sort(key=get_sort_key)

print(all_matches[0])
len(all_matches)

Using dataset directory: ipl-2008
('ipl-2008/335982.json', {'meta': {'data_version': '1.0.0', 'created': '2011-05-06', 'revision': 2}, 'info': {'balls_per_over': 6, 'city': 'Bangalore', 'dates': ['2008-04-18'], 'event': {'match_number': 1, 'name': 'Indian Premier League'}, 'gender': 'male', 'match_type': 'T20', 'officials': {'match_referees': ['J Srinath'], 'reserve_umpires': ['VN Kulkarni'], 'tv_umpires': ['AM Saheba'], 'umpires': ['Asad Rauf', 'RE Koertzen']}, 'outcome': {'by': {'runs': 140}, 'winner': 'Kolkata Knight Riders'}, 'overs': 20, 'player_of_match': ['BB McCullum'], 'players': {'Kolkata Knight Riders': ['SC Ganguly', 'BB McCullum', 'RT Ponting', 'DJ Hussey', 'Mohammad Hafeez', 'LR Shukla', 'WP Saha', 'AB Agarkar', 'AB Dinda', 'M Kartik', 'I Sharma'], 'Royal Challengers Bangalore': ['R Dravid', 'W Jaffer', 'V Kohli', 'JH Kallis', 'CL White', 'MV Boucher', 'B Akhil', 'AA Noffke', 'P Kumar', 'Z Khan', 'SB Joshi']}, 'registry': {'people': {'AA Noffke': 'b69e69ed', 'AB Agarkar':

61

In [5]:


# Group matches by season to assign logical IDs
season_matches = {}
for filepath, data in all_matches:
    season = data.get('info', {}).get('season')
    season_year = get_season_year(season)
    if season_year not in season_matches:
        season_matches[season_year] = []
    season_matches[season_year].append((filepath, data))

print(len(season_matches))

# Generate the mapping from filepath to logical match_id
filepath_to_id = {}
for season_year, matches in season_matches.items():
    for idx, (filepath, data) in enumerate(matches):
        match_id = f"IPL_{season_year}_{idx+1:03d}"
        filepath_to_id[filepath] = match_id

print(f"Sorted and grouped {len(filepath_to_id)} matches across {len(season_matches)} seasons.")
print("Example match ID mapping:", list(filepath_to_id.items())[0])

2
Sorted and grouped 61 matches across 2 seasons.
Example match ID mapping: ('ipl-2008/335982.json', 'IPL_2008_001')


In [6]:
def get_team_abbr(team_name):
    abbrev_map = {
        "Kolkata Knight Riders": "KKR",
        "Royal Challengers Bangalore": "RCB",
        "Royal Challengers Bengaluru": "RCB",
        "Chennai Super Kings": "CSK",
        "Mumbai Indians": "MI",
        "Rajasthan Royals": "RR",
        "Kings XI Punjab": "KXIP",
        "Punjab Kings": "PBKS",
        "Delhi Daredevils": "DD",
        "Delhi Capitals": "DC",
        "Deccan Chargers": "DC",
        "Sunrisers Hyderabad": "SRH",
        "Pune Warriors": "PWI",
        "Kochi Tuskers Kerala": "KTK",
        "Gujarat Lions": "GL",
        "Rising Pune Supergiant": "RPS",
        "Rising Pune Supergiants": "RPS",
        "Lucknow Super Giants": "LSG",
        "Gujarat Titans": "GT"
    }
    if team_name in abbrev_map:
        return abbrev_map[team_name]
    # Fallback to initials
    words = team_name.replace("Supergiant", "Super giant").split()
    return "".join(word[0].upper() for word in words if word[0].isalnum())

def get_ordinal(n):
    if 11 <= (n % 100) <= 13:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f"{n}{suffix}"

def calculate_overs(overs_list):
    if not overs_list:
        return 0.0
    last_over = overs_list[-1]
    over_num = last_over.get('over', 0)
    legal_balls = 0
    for d in last_over.get('deliveries', []):
        extras = d.get('extras', {})
        if 'wides' not in extras and 'noballs' not in extras:
            legal_balls += 1
    if legal_balls >= 6:
        return float(over_num + 1)
    else:
        return over_num + (legal_balls / 10.0)

def make_match_summary_text(info, winner, win_by_runs, win_by_wickets, match_id):
    season = info.get('season')
    season_year = get_season_year(season)
    date = info.get('dates', [None])[0]
    venue = info.get('venue')
    city = info.get('city', info.get('venue', 'Unknown').split(',')[0])
    teams = info.get('teams', [])
    toss_winner = info.get('toss', {}).get('winner')
    toss_decision = info.get('toss', {}).get('decision')
    pom = info.get('player_of_match', [''])[0]
    
    match_num = info.get('event', {}).get('match_number')
    match_num_str = f"Match {match_num}" if match_num else "Match"
    
    summary_parts = []
    summary_parts.append(f"IPL {season_year} match (ID: {match_id}) was played on {date} at {venue}, {city} between {teams[0]} and {teams[1]}.")
    if toss_winner and toss_decision:
        summary_parts.append(f"{toss_winner} won the toss and elected to {toss_decision}.")
        
    if winner == 'No Result':
        summary_parts.append("The match ended in a No Result.")
    elif winner:
        margin_str = ""
        if win_by_runs:
            margin_str = f"by {win_by_runs} runs"
        elif win_by_wickets:
            margin_str = f"by {win_by_wickets} wickets"
        else:
            margin_str = "via Super Over"
        summary_parts.append(f"{winner} defeated {teams[0] if winner == teams[1] else teams[1]} {margin_str}.")
        
    if pom:
        summary_parts.append(f"{pom} was named Player of the Match.")
        
    return " ".join(summary_parts)

def make_match_narrative(info, winner, win_by_runs, win_by_wickets, match_id, innings_summaries):
    season = info.get('season')
    season_year = get_season_year(season)
    venue = info.get('venue')
    city = info.get('city', info.get('venue', 'Unknown').split(',')[0])
    teams = info.get('teams', [])
    toss_winner = info.get('toss', {}).get('winner')
    toss_decision = info.get('toss', {}).get('decision')
    pom = info.get('player_of_match', [''])[0]
    
    match_num = info.get('event', {}).get('match_number')
    match_num_str = f"Match {match_num}" if match_num else "Match"
    
    narrative_parts = []
    narrative_parts.append(f"IPL {season_year} {match_num_str} was played at {venue}, {city} between {teams[0]} and {teams[1]}.")
    if toss_winner and toss_decision:
        narrative_parts.append(f"{toss_winner} won the toss and elected to {toss_decision}.")
        
    for i_summary in innings_summaries:
        team_name = i_summary['team']
        runs = i_summary['runs']
        wickets = i_summary['wickets']
        overs = i_summary['overs']
        top_scorers = i_summary['top_scorers']
        
        team_action = "posted" if i_summary['innings'] == 1 else "scored"
        top_scorer_part = ""
        if top_scorers:
            top_scorer_part = f" thanks to {top_scorers[0]['player']}'s score of {top_scorers[0]['runs']}"
            
        narrative_parts.append(f"{team_name} {team_action} {runs}/{wickets} in {overs} overs{top_scorer_part}.")
        
    if winner == 'No Result':
        narrative_parts.append("The match ended in a No Result.")
    elif winner:
        margin_str = ""
        if win_by_runs:
            margin_str = f"by {win_by_runs} runs"
        elif win_by_wickets:
            margin_str = f"by {win_by_wickets} wickets"
        else:
            margin_str = "via Super Over"
        
        narrative_parts.append(f"{winner} won the match {margin_str}.")
        
    if pom:
        narrative_parts.append(f"{pom} was named Player of the Match.")
        
    return " ".join(narrative_parts)

In [7]:
def generate_match_chunks(file_path, match_id):
    with open(file_path, 'r') as f:
        data = json.load(f)
        
    info = data.get('info', {})
    season = info.get('season', 'Unknown')
    season_year = get_season_year(season)
    
    dates = info.get('dates', [])
    date_str = dates[0] if dates else 'Unknown'
    year_val = int(date_str.split('-')[0]) if dates else 0
    
    venue = info.get('venue', 'Unknown')
    city = info.get('city', info.get('venue', 'Unknown').split(',')[0])
    
    teams = info.get('teams', [])
    if len(teams) < 2:
        return []
        
    team1, team2 = teams[0], teams[1]
    team1_abbr = get_team_abbr(team1)
    team2_abbr = get_team_abbr(team2)
    
    toss = info.get('toss', {})
    toss_winner = toss.get('winner', 'Unknown')
    toss_decision = toss.get('decision', 'Unknown').capitalize()
    
    outcome = info.get('outcome', {})
    winner = outcome.get('winner') or outcome.get('eliminator')
    if winner:
        winner_abbr = get_team_abbr(winner)
    elif outcome.get('result') == 'no result':
        winner = 'No Result'
        winner_abbr = 'No Result'
    else:
        winner = 'Draw'
        winner_abbr = 'Draw'
        
    outcome_by = outcome.get('by', {})
    win_by_runs = outcome_by.get('runs', 0)
    win_by_wickets = outcome_by.get('wickets', 0)
    
    poms = info.get('player_of_match', [])
    player_of_match = poms[0] if poms else 'None'
    
    # Common Metadata to store with every chunk
    metadata = {
        "match_id": match_id,
        "season": str(season),
        "year": year_val,
        "team1": team1_abbr,
        "team2": team2_abbr,
        "venue": venue,
        "city": city,
        "winner": winner_abbr,
        "player_of_match": player_of_match
    }
    
    chunks = []
    
    # 1. Match Summary Chunk
    match_summary_text = make_match_summary_text(info, winner, win_by_runs, win_by_wickets, match_id)
    win_str = ""
    if win_by_runs:
        win_str = f"win_by_runs: {win_by_runs}"
    elif win_by_wickets:
        win_str = f"win_by_wickets: {win_by_wickets}"
    else:
        win_str = "win_by: Super Over / Tie"
        
    match_summary_doc = f"""[Match Summary]
Match ID: {match_id}
Season: {season}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Toss: {toss_winner} won the toss and elected to {toss_decision}
Winner: {winner} ({win_str})
Player of the Match: {player_of_match}
Summary: {match_summary_text}"""

    match_summary_meta = metadata.copy()
    match_summary_meta["chunk_type"] = "match_summary"
    
    chunks.append({
        "id": f"{match_id}_match_summary",
        "document": match_summary_doc,
        "metadata": match_summary_meta
    })
    
    # Process Innings
    innings_data = data.get('innings', [])
    innings_summaries = []
    
    batting_stats = {}
    bowling_stats = {}
    match_partnerships = []
    match_milestones = []
    match_wickets = []
    
    for inning_idx, inning in enumerate(innings_data):
        batting_team = inning.get('team')
        bowling_team = team2 if batting_team == team1 else team1
        batting_team_abbr = get_team_abbr(batting_team)
        bowling_team_abbr = get_team_abbr(bowling_team)
        
        inning_num = inning_idx + 1
        overs_list = inning.get('overs', [])
        
        inning_runs = 0
        inning_wickets_count = 0
        
        current_partnership_batsmen = None
        part_runs = 0
        part_balls = 0
        
        reached_50 = set()
        reached_100 = set()
        reached_150 = set()
        
        bowler_wickets_in_innings = {}
        reached_3w = set()
        reached_5w = set()
        
        for over_obj in overs_list:
            over_index = over_obj.get('over', 0)
            deliveries = over_obj.get('deliveries', [])
            
            legal_balls = 0
            for delivery in deliveries:
                striker = delivery['batter']
                non_striker = delivery['non_striker']
                bowler = delivery['bowler']
                runs_dict = delivery.get('runs', {})
                batter_runs = runs_dict.get('batter', 0)
                extras_runs = runs_dict.get('extras', 0)
                total_runs = runs_dict.get('total', 0)
                
                extras = delivery.get('extras', {})
                is_wide = 'wides' in extras
                is_noball = 'noballs' in extras
                is_legal = not is_wide and not is_noball
                
                if is_legal:
                    legal_balls += 1
                
                ball_display = legal_balls if is_legal else (legal_balls + 1)
                over_val = over_index + (ball_display / 10.0)
                
                inning_runs += total_runs
                
                if striker not in batting_stats:
                    batting_stats[striker] = {
                        "runs": 0, "balls": 0, "fours": 0, "sixes": 0,
                        "team": batting_team, "opposing_team": bowling_team,
                        "dismissed": False, "how_out": "not out", "bowler": None, "fielder": None
                    }
                if non_striker not in batting_stats:
                    batting_stats[non_striker] = {
                        "runs": 0, "balls": 0, "fours": 0, "sixes": 0,
                        "team": batting_team, "opposing_team": bowling_team,
                        "dismissed": False, "how_out": "not out", "bowler": None, "fielder": None
                    }
                if bowler not in bowling_stats:
                    bowling_stats[bowler] = {
                        "wickets": 0, "runs_conceded": 0, "balls_bowled": 0,
                        "team": bowling_team, "opposing_team": batting_team
                    }
                
                batting_stats[striker]["runs"] += batter_runs
                if is_legal:
                    batting_stats[striker]["balls"] += 1
                if batter_runs == 4:
                    batting_stats[striker]["fours"] += 1
                elif batter_runs == 6:
                    batting_stats[striker]["sixes"] += 1
                    
                conceded = total_runs - extras.get('byes', 0) - extras.get('legbyes', 0) - extras.get('penalty', 0)
                conceded = max(0, conceded)
                bowling_stats[bowler]["runs_conceded"] += conceded
                if is_legal:
                    bowling_stats[bowler]["balls_bowled"] += 1
                    
                if current_partnership_batsmen is None:
                    current_partnership_batsmen = [striker, non_striker]
                    part_runs = 0
                    part_balls = 0
                    
                part_runs += total_runs
                if is_legal:
                    part_balls += 1
                    
                # Milestones
                current_batsman_runs = batting_stats[striker]["runs"]
                current_batsman_balls = batting_stats[striker]["balls"]
                
                if current_batsman_runs >= 50 and striker not in reached_50:
                    reached_50.add(striker)
                    match_milestones.append({
                        "player": striker,
                        "milestone": "Half-Century",
                        "runs": 50,
                        "over": over_val,
                        "summary": f"{striker} reached his half-century in {current_batsman_balls} balls against {bowling_team}."
                    })
                if current_batsman_runs >= 100 and striker not in reached_100:
                    reached_100.add(striker)
                    match_milestones.append({
                        "player": striker,
                        "milestone": "Century",
                        "runs": 100,
                        "over": over_val,
                        "summary": f"{striker} reached his century in {current_batsman_balls} balls against {bowling_team}."
                    })
                if current_batsman_runs >= 150 and striker not in reached_150:
                    reached_150.add(striker)
                    match_milestones.append({
                        "player": striker,
                        "milestone": "150 Runs",
                        "runs": 150,
                        "over": over_val,
                        "summary": f"{striker} reached 150 runs in {current_batsman_balls} balls against {bowling_team}."
                    })
                
                # Wickets
                wickets_list = delivery.get('wickets', [])
                if wickets_list:
                    for w in wickets_list:
                        player_out = w.get('player_out')
                        dismissal_kind = w.get('kind', 'out')
                        fielders_list = w.get('fielders', [])
                        fielders_names = [fld.get('name') for fld in fielders_list if fld.get('name')]
                        fielders_str = " and ".join(fielders_names) if fielders_names else None
                        
                        inning_wickets_count += 1
                        
                        if player_out in batting_stats:
                            batting_stats[player_out]["dismissed"] = True
                            batting_stats[player_out]["how_out"] = dismissal_kind
                            batting_stats[player_out]["bowler"] = bowler
                            batting_stats[player_out]["fielder"] = fielders_str
                            
                        is_bowler_wicket = dismissal_kind in ['caught', 'bowled', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']
                        if is_bowler_wicket:
                            bowling_stats[bowler]["wickets"] += 1
                            bowler_wickets_in_innings[bowler] = bowler_wickets_in_innings.get(bowler, 0) + 1
                            current_w = bowler_wickets_in_innings[bowler]
                            
                            if current_w == 3 and bowler not in reached_3w:
                                reached_3w.add(bowler)
                                match_milestones.append({
                                    "player": bowler,
                                    "milestone": "3 Wicket Haul",
                                    "runs": 3,
                                    "over": over_val,
                                    "summary": f"{bowler} completed a 3-wicket haul ({current_w}/{bowling_stats[bowler]['runs_conceded']}) at {over_val} overs."
                                })
                            if current_w == 5 and bowler not in reached_5w:
                                reached_5w.add(bowler)
                                match_milestones.append({
                                    "player": bowler,
                                    "milestone": "5 Wicket Haul",
                                    "runs": 5,
                                    "over": over_val,
                                    "summary": f"{bowler} completed a 5-wicket haul ({current_w}/{bowling_stats[bowler]['runs_conceded']}) at {over_val} overs."
                                })
                                
                        match_wickets.append({
                            "over": over_val,
                            "batsman": player_out,
                            "bowler": bowler,
                            "dismissal": dismissal_kind,
                            "fielder": fielders_str,
                            "inning_num": inning_num,
                            "batting_team": batting_team
                        })
                        
                        p_label = "Opening" if inning_wickets_count == 1 else f"{get_ordinal(inning_wickets_count)} wicket"
                        p_summary = f"{p_label} partnership between {current_partnership_batsmen[0]} and {current_partnership_batsmen[1]} added {part_runs} runs."
                        match_partnerships.append({
                            "players": list(current_partnership_batsmen),
                            "runs": part_runs,
                            "wicket": inning_wickets_count,
                            "unbroken": False,
                            "summary": p_summary,
                            "team": batting_team,
                            "inning_num": inning_num
                        })
                        
                        current_partnership_batsmen = None
                        part_runs = 0
                        part_balls = 0
                        
        if current_partnership_batsmen is not None:
            p_summary = f"Unbroken partnership of {part_runs} runs between {current_partnership_batsmen[0]} and {current_partnership_batsmen[1]}."
            match_partnerships.append({
                "players": list(current_partnership_batsmen),
                "runs": part_runs,
                "wicket": inning_wickets_count + 1,
                "unbroken": True,
                "summary": p_summary,
                "team": batting_team,
                "inning_num": inning_num
            })
            
        overs_bowled = calculate_overs(overs_list)
        
        inning_batsmen = {p: stats for p, stats in batting_stats.items() if stats["team"] == batting_team}
        sorted_batsmen = sorted(inning_batsmen.items(), key=lambda x: x[1]["runs"], reverse=True)
        top_scorers = []
        for player, stats in sorted_batsmen[:2]:
            top_scorers.append({
                "player": player,
                "runs": stats["runs"]
            })
            
        top_scorer_part = ""
        if top_scorers:
            top1_player = top_scorers[0]["player"]
            top1_runs = top_scorers[0]["runs"]
            is_unbeaten = not batting_stats[top1_player]["dismissed"]
            status_verb = "remained unbeaten on" if is_unbeaten else "scored"
            top_scorer_part = f" {top1_player} {status_verb} {top1_runs} runs."
            
        innings_summary_text = f"{batting_team_abbr} scored {inning_runs}/{inning_wickets_count} in {overs_bowled} overs.{top_scorer_part}"
        
        innings_summaries.append({
            "innings": inning_num,
            "team": batting_team,
            "runs": inning_runs,
            "wickets": inning_wickets_count,
            "overs": overs_bowled,
            "top_scorers": top_scorers,
            "summary": innings_summary_text
        })
        
        inning_summary_doc = f"""[Team Innings Summary]
Match ID: {match_id}
Season/Year: {season_year}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Team: {batting_team}
Innings: {inning_num}
Score: {inning_runs}/{inning_wickets_count} in {overs_bowled} overs
Top Scorers:
""" + "\n".join([f"- {ts['player']}: {ts['runs']} runs" for ts in top_scorers]) + f"\nSummary: {innings_summary_text} (Match played between {team1} and {team2} on {date_str})"

        inning_summary_meta = metadata.copy()
        inning_summary_meta["chunk_type"] = "innings_summary"
        
        chunks.append({
            "id": f"{match_id}_innings_{inning_num}",
            "document": inning_summary_doc,
            "metadata": inning_summary_meta
        })

    # Player Batting Chunks
    for player, stats in batting_stats.items():
        runs = stats["runs"]
        balls = stats["balls"]
        fours = stats["fours"]
        sixes = stats["sixes"]
        team = stats["team"]
        opp_team = stats["opposing_team"]
        dismissed = stats["dismissed"]
        strike_rate = round((runs / balls * 100), 1) if balls > 0 else 0.0
        
        not_out_suffix = "not out" if not dismissed else "out"
        how_out_text = f"dismissed ({stats['how_out']})" if dismissed else "not out"
        player_batting_text = f"{player} scored {runs} {not_out_suffix} off {balls} balls against {opp_team} in IPL {season_year}."
        
        batting_doc = f"""[Player Batting Performance]
Match ID: {match_id}
Season/Year: {season_year}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Player: {player}
Team: {team}
Opponent: {opp_team}
Runs: {runs} | Balls: {balls} | Fours: {fours} | Sixes: {sixes} | SR: {strike_rate}
Status: {how_out_text}
Summary: {player_batting_text}"""

        batting_meta = metadata.copy()
        batting_meta["chunk_type"] = "player_batting"
        
        chunks.append({
            "id": f"{match_id}_batting_{player.replace(' ', '_')}",
            "document": batting_doc,
            "metadata": batting_meta
        })

    # Player Bowling Chunks
    for player, stats in bowling_stats.items():
        wickets = stats["wickets"]
        runs_conceded = stats["runs_conceded"]
        balls_bowled = stats["balls_bowled"]
        team = stats["team"]
        opp_team = stats["opposing_team"]
        
        overs_bowled = (balls_bowled // 6) + (balls_bowled % 6) / 10.0
        overs_bowled_fractional = balls_bowled / 6.0
        economy = round(runs_conceded / overs_bowled_fractional, 2) if balls_bowled > 0 else 0.0
        
        overs_str = f"{balls_bowled // 6}" if balls_bowled % 6 == 0 else f"{balls_bowled // 6}.{balls_bowled % 6}"
        player_bowling_text = f"{player} took {wickets} wickets conceding {runs_conceded} runs in {overs_str} overs (economy {economy}) against {opp_team} in IPL {season_year}."
        
        bowling_doc = f"""[Player Bowling Performance]
Match ID: {match_id}
Season/Year: {season_year}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Player: {player}
Team: {team}
Opponent: {opp_team}
Overs: {overs_str} | Wickets: {wickets} | Runs Conceded: {runs_conceded} | Economy: {economy}
Summary: {player_bowling_text}"""

        bowling_meta = metadata.copy()
        bowling_meta["chunk_type"] = "player_bowling"
        
        chunks.append({
            "id": f"{match_id}_bowling_{player.replace(' ', '_')}",
            "document": bowling_doc,
            "metadata": bowling_meta
        })

    # Partnership Chunks
    for part_idx, part in enumerate(match_partnerships):
        players = part["players"]
        runs = part["runs"]
        wicket = part["wicket"]
        unbroken = part["unbroken"]
        team = part["team"]
        inning_num = part["inning_num"]
        
        part_doc = f"""[Partnership]
Match ID: {match_id}
Season/Year: {season_year}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Team: {team}
Innings: {inning_num}
Players: {players[0]} & {players[1]}
Wicket: {wicket}
Runs: {runs}
Summary: {part['summary']} (Match played between {team1} and {team2} on {date_str})"""

        part_meta = metadata.copy()
        part_meta["chunk_type"] = "partnership"
        
        chunks.append({
            "id": f"{match_id}_partnership_inning{inning_num}_{part_idx}",
            "document": part_doc,
            "metadata": part_meta
        })

    # Wicket Chunks
    for wicket_idx, w_evt in enumerate(match_wickets):
        over_val = w_evt["over"]
        batsman = w_evt["batsman"]
        bowler = w_evt["bowler"]
        dismissal = w_evt["dismissal"]
        fielder = w_evt["fielder"]
        inning_num = w_evt["inning_num"]
        bat_team = w_evt["batting_team"]
        
        fielder_part = f" caught by {fielder}" if fielder and dismissal == "caught" else ""
        if dismissal == "caught" and not fielder:
            fielder_part = f" caught"
        elif dismissal == "stumped" and fielder:
            fielder_part = f" stumped by {fielder}"
            
        summary_text = ""
        if dismissal == "bowled":
            summary_text = f"{batsman} was bowled by {bowler} at {over_val} overs."
        elif dismissal in ["caught", "stumped"]:
            summary_text = f"{batsman} was{fielder_part} off {bowler} at {over_val} overs."
        elif dismissal == "lbw":
            summary_text = f"{batsman} was out LBW off {bowler} at {over_val} overs."
        elif dismissal == "run out":
            fielder_info = f" by {fielder}" if fielder else ""
            summary_text = f"{batsman} was run out{fielder_info} at {over_val} overs."
        else:
            summary_text = f"{batsman} was dismissed ({dismissal}) off {bowler} at {over_val} overs."
            
        wicket_doc = f"""[Wicket Event]
Match ID: {match_id}
Season/Year: {season_year}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Innings: {inning_num}
Batting Team: {bat_team}
Batsman Dismissed: {batsman}
Bowler: {bowler}
Dismissal Type: {dismissal}
Fielder: {fielder or 'N/A'}
Over: {over_val}
Summary: {summary_text} (Match played between {team1} and {team2} on {date_str})"""

        wicket_meta = metadata.copy()
        wicket_meta["chunk_type"] = "wicket_event"
        
        chunks.append({
            "id": f"{match_id}_wicket_{inning_num}_{wicket_idx}",
            "document": wicket_doc,
            "metadata": wicket_meta
        })

    # Milestone Chunks
    for milestone_idx, ms in enumerate(match_milestones):
        player = ms["player"]
        milestone_name = ms["milestone"]
        runs_val = ms["runs"]
        over_val = ms["over"]
        
        ms_doc = f"""[Milestone]
Match ID: {match_id}
Season/Year: {season_year}
Date: {date_str}
Venue: {venue}, {city}
Teams: {team1} vs {team2}
Player: {player}
Milestone: {milestone_name}
Value: {runs_val}
Over: {over_val}
Summary: {ms['summary']} (Match played between {team1} and {team2} on {date_str})"""

        ms_meta = metadata.copy()
        ms_meta["chunk_type"] = "milestone"
        
        chunks.append({
            "id": f"{match_id}_milestone_{milestone_idx}",
            "document": ms_doc,
            "metadata": ms_meta
        })

    # 8. Match Narrative Chunk
    narrative_text = make_match_narrative(info, winner, win_by_runs, win_by_wickets, match_id, innings_summaries)
    
    narrative_doc = f"""[Match Narrative]
Match ID: {match_id}
Narrative: {narrative_text}"""

    narrative_meta = metadata.copy()
    narrative_meta["chunk_type"] = "match_narrative"
    
    chunks.append({
        "id": f"{match_id}_narrative",
        "document": narrative_doc,
        "metadata": narrative_meta
    })

    return chunks

In [8]:
# Test chunking function on a sample match
import os
sample_file = f"{dataset_dir}/335982.json"
if not os.path.exists(sample_file):
    # Fallback to the first available match file
    import glob
    available_files = glob.glob(f'{dataset_dir}/*.json')
    sample_file = available_files[0] if available_files else None

sample_id = filepath_to_id[sample_file]

sample_chunks = generate_match_chunks(sample_file, sample_id)
# print(f"Generated {len(sample_chunks)} chunks for {sample_id}.")

# print("\n--- Sample Match Summary Chunk ---")
# print(sample_chunks[0]['document'])
# print("Metadata:", sample_chunks[0]['metadata'])

# print("\n--- Sample Innings 1 Summary Chunk ---")
# print(sample_chunks[1]['document'])

# Let's inspect some other chunk types
for chunk in sample_chunks:
    ctype = chunk['metadata']['chunk_type']
    print(f"\n--- Sample {ctype} Chunk ---")
    print(chunk['document'])


--- Sample match_summary Chunk ---
[Match Summary]
Match ID: IPL_2008_001
Season: 2007/08
Date: 2008-04-18
Venue: M Chinnaswamy Stadium, Bangalore
Teams: Royal Challengers Bangalore vs Kolkata Knight Riders
Toss: Royal Challengers Bangalore won the toss and elected to Field
Winner: Kolkata Knight Riders (win_by_runs: 140)
Player of the Match: BB McCullum
Summary: IPL 2008 match (ID: IPL_2008_001) was played on 2008-04-18 at M Chinnaswamy Stadium, Bangalore between Royal Challengers Bangalore and Kolkata Knight Riders. Royal Challengers Bangalore won the toss and elected to field. Kolkata Knight Riders defeated Royal Challengers Bangalore by 140 runs. BB McCullum was named Player of the Match.

--- Sample innings_summary Chunk ---
[Team Innings Summary]
Match ID: IPL_2008_001
Season/Year: 2008
Date: 2008-04-18
Venue: M Chinnaswamy Stadium, Bangalore
Teams: Royal Challengers Bangalore vs Kolkata Knight Riders
Team: Kolkata Knight Riders
Innings: 1
Score: 222/3 in 20.0 overs
Top Scorers:

In [9]:
import tqdm

all_chunks = []
for filepath, match_id in tqdm.tqdm(filepath_to_id.items(), desc="Generating chunks for all matches"):
    match_chunks = generate_match_chunks(filepath, match_id)
    all_chunks.extend(match_chunks)

print(f"Total chunks generated: {len(all_chunks)}")

# Print count by chunk type
chunk_type_counts = {}
for chunk in all_chunks:
    ctype = chunk['metadata']['chunk_type']
    chunk_type_counts[ctype] = chunk_type_counts.get(ctype, 0) + 1

print("\nChunk count breakdown:")
for ctype, count in chunk_type_counts.items():
    print(f"  - {ctype}: {count} chunks")

Generating chunks for all matches: 100%|██████████| 61/61 [00:00<00:00, 638.04it/s]

Total chunks generated: 3611

Chunk count breakdown:
  - match_summary: 61 chunks
  - innings_summary: 122 chunks
  - player_batting: 944 chunks
  - player_bowling: 717 chunks
  - partnership: 822 chunks
  - wicket_event: 729 chunks
  - milestone: 155 chunks
  - match_narrative: 61 chunks


In [10]:
client = chromadb.PersistentClient(path="./chroma_db")

# Use OpenAI's text-embedding-3-large embedding function
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name=embedding_model_name
)

# Get or create collection dynamically to avoid already-exists or locking errors
collection_name = "ipl_enhanced_chunks_openai"
collection = client.create_collection(
    name=collection_name,
    embedding_function=openai_ef,
    get_or_create=True
)

# Delete existing documents inside the collection to start fresh
try:
    existing_ids = collection.get(include=[])['ids']
    if existing_ids:
        batch_size = 1000
        for i in range(0, len(existing_ids), batch_size):
            collection.delete(ids=existing_ids[i:i+batch_size])
        print(f"Cleared {len(existing_ids)} existing documents from collection '{collection_name}' to start fresh.")
    else:
        print(f"Collection '{collection_name}' is already empty and ready.")
except Exception as e:
    print(f"Warning during clearing collection: {e}")

Cleared 3611 existing documents from collection 'ipl_enhanced_chunks_openai' to start fresh.


In [11]:
def safe_add_to_collection(collection, client, collection_name, openai_ef, docs, metas, ids, retries=5, delay=5):
    last_err = None
    for attempt in range(retries):
        try:
            collection.add(documents=docs, metadatas=metas, ids=ids)
            return collection, client
        except Exception as e:
            last_err = e
            print(f"\nError during ingestion (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                import time
                time.sleep(delay)
                try:
                    client = chromadb.PersistentClient(path="./chroma_db")
                    collection = client.get_collection(name=collection_name, embedding_function=openai_ef)
                except Exception as re_err:
                    print(f"Failed to re-initialize client: {re_err}")
            else:
                print("Max retries reached. Raising error.")
                raise last_err

# Ingest in batches of 1000
batch_size = 1000
total_chunks = len(all_chunks)

for i in range(0, total_chunks, batch_size):
    batch = all_chunks[i:i+batch_size]
    batch_docs = [chunk['document'] for chunk in batch]
    batch_metas = [chunk['metadata'] for chunk in batch]
    batch_ids = [chunk['id'] for chunk in batch]
    
    collection, client = safe_add_to_collection(
        collection=collection,
        client=client,
        collection_name=collection_name,
        openai_ef=openai_ef,
        docs=batch_docs,
        metas=batch_metas,
        ids=batch_ids
    )
    print(f"Ingested chunks {i+1} to {min(i+batch_size, total_chunks)} of {total_chunks}...")

print(f"Successfully loaded {total_chunks} chunks into ChromaDB collection '{collection_name}'!")

Ingested chunks 1 to 1000 of 3611...
Ingested chunks 1001 to 2000 of 3611...
Ingested chunks 2001 to 3000 of 3611...
Ingested chunks 3001 to 3611 of 3611...
Successfully loaded 3611 chunks into ChromaDB collection 'ipl_enhanced_chunks_openai'!


In [12]:
class Result:
    page_content: str
    metadata: dict

class Result:
    def __init__(self, page_content: str, metadata: dict):
        self.page_content = page_content
        self.metadata = metadata

    def __repr__(self):
        return f"Result(page_content={self.page_content!r}, metadata={self.metadata!r})"

def fetch_context(question):
    query = openai.embeddings.create(model=embedding_model_name, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=20)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [62]:
chunks=fetch_context("Royal Challengers Bangalore won the matches")
for chunk in chunks:
    print(chunk)
    print()


Result(page_content='[Match Summary]\nMatch ID: IPL_2008_012\nSeason: 2007/08\nDate: 2008-04-26\nVenue: M Chinnaswamy Stadium, Bangalore\nTeams: Royal Challengers Bangalore vs Rajasthan Royals\nToss: Rajasthan Royals won the toss and elected to Field\nWinner: Rajasthan Royals (win_by_wickets: 7)\nPlayer of the Match: SR Watson\nSummary: IPL 2008 match (ID: IPL_2008_012) was played on 2008-04-26 at M Chinnaswamy Stadium, Bangalore between Royal Challengers Bangalore and Rajasthan Royals. Rajasthan Royals won the toss and elected to field. Rajasthan Royals defeated Royal Challengers Bangalore by 7 wickets. SR Watson was named Player of the Match.', metadata={'player_of_match': 'SR Watson', 'year': 2008, 'winner': 'RR', 'venue': 'M Chinnaswamy Stadium', 'team2': 'RR', 'chunk_type': 'match_summary', 'match_id': 'IPL_2008_012', 'season': '2007/08', 'team1': 'RCB', 'city': 'Bangalore'})

Result(page_content='[Match Summary]\nMatch ID: IPL_2008_050\nSeason: 2007/08\nDate: 2008-05-25\nVenue: R

In [13]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing IPL cricket maches data.
You are chatting with a user about IPL.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""



In [14]:
MODEL = "gpt-4.1-nano"

In [15]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the IPL (i.e Cricket League)Season.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the IPL name unless it's a general question about the IPL.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [16]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['match_id']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [17]:
print(fetch_context("highest partenership score in 2008"))

[Result(page_content='[Partnership]\nMatch ID: IPL_2008_034\nSeason/Year: 2008\nDate: 2008-05-12\nVenue: Punjab Cricket Association Stadium, Mohali, Chandigarh\nTeams: Kings XI Punjab vs Royal Challengers Bangalore\nTeam: Kings XI Punjab\nInnings: 2\nPlayers: SE Marsh & LA Pomersbach\nWicket: 2\nRuns: 102\nSummary: Unbroken partnership of 102 runs between SE Marsh and LA Pomersbach. (Match played between Kings XI Punjab and Royal Challengers Bangalore on 2008-05-12)', metadata={'team1': 'KXIP', 'venue': 'Punjab Cricket Association Stadium, Mohali', 'team2': 'RCB', 'winner': 'KXIP', 'city': 'Chandigarh', 'chunk_type': 'partnership', 'player_of_match': 'SE Marsh', 'year': 2008, 'season': '2007/08', 'match_id': 'IPL_2008_034'}), Result(page_content='[Partnership]\nMatch ID: IPL_2008_005\nSeason/Year: 2008\nDate: 2008-04-20\nVenue: Wankhede Stadium, Mumbai\nTeams: Mumbai Indians vs Royal Challengers Bangalore\nTeam: Royal Challengers Bangalore\nInnings: 2\nPlayers: JH Kallis & LRPL Taylor\

In [18]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query= rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    print (chunks)
    messages = make_rag_messages(query, history, chunks)
    print (messages)

    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [19]:
answer_question("to 10 highest score partenership score in 2008 ", [])

What are the top 10 highest partnership scores in 2008?
[Result(page_content='[Partnership]\nMatch ID: IPL_2008_034\nSeason/Year: 2008\nDate: 2008-05-12\nVenue: Punjab Cricket Association Stadium, Mohali, Chandigarh\nTeams: Kings XI Punjab vs Royal Challengers Bangalore\nTeam: Kings XI Punjab\nInnings: 2\nPlayers: SE Marsh & LA Pomersbach\nWicket: 2\nRuns: 102\nSummary: Unbroken partnership of 102 runs between SE Marsh and LA Pomersbach. (Match played between Kings XI Punjab and Royal Challengers Bangalore on 2008-05-12)', metadata={'year': 2008, 'venue': 'Punjab Cricket Association Stadium, Mohali', 'match_id': 'IPL_2008_034', 'winner': 'KXIP', 'city': 'Chandigarh', 'team1': 'KXIP', 'team2': 'RCB', 'chunk_type': 'partnership', 'season': '2007/08', 'player_of_match': 'SE Marsh'}), Result(page_content='[Partnership]\nMatch ID: IPL_2008_050\nSeason/Year: 2008\nDate: 2008-05-25\nVenue: Rajiv Gandhi International Stadium, Uppal, Hyderabad\nTeams: Deccan Chargers vs Royal Challengers Bangal

'Based on the available data for the 2008 IPL season, the top 10 highest partnership scores are:\n\n1. 155 runs between AC Gilchrist & VVS Laxman (Match IPL_2008_014, April 27, 2008)\n2. 133 runs between LA Pomersbach & SE Marsh (Match IPL_2008_045, May 21, 2008)\n3. 133 runs between SE Marsh & JR Hopes (Match IPL_2008_055, May 28, 2008)\n4. 109 runs between GC Smith & SA Asnodkar (Match IPL_2008_039, May 17, 2008)\n5. 102 runs between PA Patel & SK Raina (Match IPL_2008_057, May 31, 2008)\n6. 102 runs between SE Marsh & LA Pomersbach (Match IPL_2008_034, May 12, 2008)\n7. 101 runs between AC Gilchrist & HH Gibbs (Match IPL_2008_050, May 25, 2008)\n8. 90 runs between G Gambhir & V Sehwag (Match IPL_2008_043, May 19, 2008)\n9. 84 runs between SE Marsh & KC Sangakkara (Match IPL_2008_047, May 23, 2008)\n10. 79 runs between AC Gilchrist & HH Gibbs (Match IPL_2008_047, May 23, 2008)\n\nPlease note that the exact rankings are based solely on the partnership runs documented in the data provi

In [20]:
def test_query(query_text, ctype_filter=None, n=3):
    print(f"\nQuery: '{query_text}'" + (f" (Filter: {ctype_filter})" if ctype_filter else ""))
    
    kwargs = {"query_texts": [query_text], "n_results": n}
    if ctype_filter:
        kwargs["where"] = {"chunk_type": ctype_filter}
        
    results = collection.query(**kwargs)
    
    for idx in range(len(results['ids'][0])):
        print(f"  [{idx+1}] ID: {results['ids'][0][idx]} (Distance: {results['distances'][0][idx]:.4f})")
        # Print first few lines of the retrieved document
        doc_preview = "\n".join(results['documents'][0][idx].split('\n')[:10])
        print(f"  Document Preview:\n{doc_preview}\n")
        print("-" * 50)

# 1. Find all matches won by KKR in Bangalore
test_query("KKR won in Bangalore", ctype_filter="match_summary")

# 2. How was Ganguly dismissed in the first IPL match
test_query("Ganguly dismissal in first IPL match 2008", ctype_filter="wicket_event")

# 3. Brendon McCullum performance opening match
test_query("Brendon McCullum 158 runs opening match", ctype_filter="player_batting")


Query: 'KKR won in Bangalore' (Filter: match_summary)
  [1] ID: IPL_2008_001_match_summary (Distance: 0.3779)
  Document Preview:
[Match Summary]
Match ID: IPL_2008_001
Season: 2007/08
Date: 2008-04-18
Venue: M Chinnaswamy Stadium, Bangalore
Teams: Royal Challengers Bangalore vs Kolkata Knight Riders
Toss: Royal Challengers Bangalore won the toss and elected to Field
Winner: Kolkata Knight Riders (win_by_runs: 140)
Player of the Match: BB McCullum
Summary: IPL 2008 match (ID: IPL_2008_001) was played on 2008-04-18 at M Chinnaswamy Stadium, Bangalore between Royal Challengers Bangalore and Kolkata Knight Riders. Royal Challengers Bangalore won the toss and elected to field. Kolkata Knight Riders defeated Royal Challengers Bangalore by 140 runs. BB McCullum was named Player of the Match.

--------------------------------------------------
  [2] ID: IPL_2008_029_match_summary (Distance: 0.3825)
  Document Preview:
[Match Summary]
Match ID: IPL_2008_029
Season: 2007/08
Date: 2008-05-08
Ve

## Visualizing Vector Embeddings of Cricket Chunks\n
\n
Since we have generated and ingested 70,000+ chunks, running t-SNE on the entire dataset at once is computationally expensive. Instead, we can retrieve a random sample of 3,000 chunks distributed across the entire dataset to inspect the vector space.\n
\n
We will color-code the chunks by their `chunk_type` to visualize how different types of cricket knowledge (match summaries, player performances, wickets, partnerships, milestones, narratives) cluster together.

In [21]:
import random
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Re-initialize client and collection to avoid stale references after collection reset
client = chromadb.PersistentClient(path="./chroma_db")
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-large"
)
collection = client.get_collection(name="ipl_enhanced_chunks_openai", embedding_function=openai_ef)

# Retrieve a representative sample of 3000 chunks
print("Retrieving sample of chunk IDs...")
all_ids = collection.get(include=[])['ids']
random.seed(42)
sample_ids = random.sample(all_ids, min(3000, len(all_ids)))

print(f"Fetching embeddings and metadata for {len(sample_ids)} sampled chunks...")
result = collection.get(ids=sample_ids, include=['embeddings', 'documents', 'metadatas'])

vectors = np.array(result['embeddings'])
metadatas = result['metadatas']

print("Running 2D t-SNE dimensional reduction...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30)
reduced_vectors_2d = tsne_2d.fit_transform(vectors)

# Define a premium color scheme for chunk types
chunk_colors = {
    "match_summary": "#00E5FF",      # Bright Cyan
    "innings_summary": "#2979FF",    # Bright Blue
    "player_batting": "#00E676",      # Vibrant Green
    "player_bowling": "#FF1744",      # Crimson Red
    "partnership": "#FF9100",         # Amber/Orange
    "wicket_event": "#D500F9",        # Neon Purple
    "milestone": "#FFD600",           # Canary Yellow
    "match_narrative": "#F50057"      # Deep Pink
}

# Group data by chunk type for Plotly trace generation
traces = []
for ctype, color in chunk_colors.items():
    indices = [i for i, m in enumerate(metadatas) if m.get('chunk_type') == ctype]
    if not indices:
        continue
        
    sub_vectors = reduced_vectors_2d[indices]
    sub_metas = [metadatas[i] for i in indices]
    
    hover_texts = []
    for m in sub_metas:
        hover_text = (
            f"<b>Type:</b> {m.get('chunk_type')}<br>"
            f"<b>Match ID:</b> {m.get('match_id')}<br>"
            f"<b>Season:</b> {m.get('season')}<br>"
            f"<b>Teams:</b> {m.get('team1')} vs {m.get('team2')}<br>"
            f"<b>Winner:</b> {m.get('winner')}<br>"
            f"<b>Venue:</b> {m.get('venue')}"
        )
        hover_texts.append(hover_text)
        
    traces.append(go.Scatter(
        x=sub_vectors[:, 0],
        y=sub_vectors[:, 1],
        mode='markers',
        name=ctype.replace('_', ' ').title(),
        marker=dict(
            size=6,
            color=color,
            opacity=0.75,
            line=dict(width=0.5, color='rgba(255,255,255,0.2)')
        ),
        text=hover_texts,
        hoverinfo='text'
    ))

fig = go.Figure(data=traces)
fig.update_layout(
    title='2D t-SNE Visualization of Cricket Knowledge Chunks in ChromaDB',
    xaxis_title='t-SNE Dimension 1',
    yaxis_title='t-SNE Dimension 2',
    width=1000,
    height=700,
    template="plotly_dark",
    legend=dict(title="Chunk Types", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

Retrieving sample of chunk IDs...
Fetching embeddings and metadata for 3000 sampled chunks...
Running 2D t-SNE dimensional reduction...


In [22]:
print("Running 3D t-SNE dimensional reduction...")
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=30)
reduced_vectors_3d = tsne_3d.fit_transform(vectors)

traces_3d = []
for ctype, color in chunk_colors.items():
    indices = [i for i, m in enumerate(metadatas) if m.get('chunk_type') == ctype]
    if not indices:
        continue
        
    sub_vectors = reduced_vectors_3d[indices]
    sub_metas = [metadatas[i] for i in indices]
    
    hover_texts = []
    for m in sub_metas:
        hover_text = (
            f"<b>Type:</b> {m.get('chunk_type')}<br>"
            f"<b>Match ID:</b> {m.get('match_id')}<br>"
            f"<b>Season:</b> {m.get('season')}<br>"
            f"<b>Teams:</b> {m.get('team1')} vs {m.get('team2')}<br>"
            f"<b>Winner:</b> {m.get('winner')}<br>"
            f"<b>Venue:</b> {m.get('venue')}"
        )
        hover_texts.append(hover_text)
        
    traces_3d.append(go.Scatter3d(
        x=sub_vectors[:, 0],
        y=sub_vectors[:, 1],
        z=sub_vectors[:, 2],
        mode='markers',
        name=ctype.replace('_', ' ').title(),
        marker=dict(
            size=4,
            color=color,
            opacity=0.75,
            line=dict(width=0.5, color='rgba(255,255,255,0.2)')
        ),
        text=hover_texts,
        hoverinfo='text'
    ))

fig_3d = go.Figure(data=traces_3d)
fig_3d.update_layout(
    title='3D t-SNE Visualization of Cricket Knowledge Chunks in ChromaDB',
    width=1000,
    height=800,
    template="plotly_dark",
    legend=dict(title="Chunk Types", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_3d.show()

Running 3D t-SNE dimensional reduction...


In [2]:
test_query("team of KKR", ctype_filter="match_summary")

NameError: name 'test_query' is not defined